В этом файле представлен код для краулинга сайта (сбора текстов).

Подключаемся к сайту http://tolstoy-lit.ru/tolstoy/proza/index.htm. На этом сайте находятся все тексты Толстого. Я выбрала этот корпус, потому что интересно посмотреть на тексты именно одного автора, на художественные произведения -- это немного другой стиль, к тому же другая эпоха. Ну и "потому что он мне понравился".

Я скачаю сначала все тексты, которые более-менее быстро скачиваются (около 8 минут), а потом оттуда выберу 4000 предложений (на самом деле записей, разделённых точкой). При желании можно было бы использовать все-все предложения, которые получаются, но таких записей будет, наверное, порядка миллиона, поиск на таком объёме достаточно долго имплементировать.

Создание корпуса и его обработка представлены в другом файле по пути ```data/preprocessing.ipynb```, здесь лишь непосредственное скачивание текстов с сайта.

# Установка и импорт необходимых библиотек

In [ ]:
# !python -m pip install fake_useragent
# !python -m pip install requests
# !python -m pip install bs4
# !python -m pip install tqdm

In [2]:
# для скачивания текстов
import requests
from fake_useragent import UserAgent
import time
import random
from bs4 import BeautifulSoup
import re
import json

# для отслеживания прогрессов
from tqdm import tqdm

# Подключение к сайту и проверка подключения

In [3]:
# начинаем сессию
session = requests.session()

In [4]:
# подключаем useragent, чтобы не приняли за бота
ua = UserAgent()

In [5]:
# подключаемся к сайте и проверяем подключение
url = "http://tolstoy-lit.ru/tolstoy/proza/index.htm"
headers = {'User-Agent': ua.random}

response = session.get(url, headers=headers)
response

<Response [200]>

# Функции для получения текстов с сайта

In [6]:
def get_the_books(url):
    '''
    Находит ссылки на тексты.
    На вход принимает ссылку на начальную страницу.
    На выходе получаем список словарей с ссылками и названиями книг.
    '''
    headers = {'User-Agent': ua.random}
    response = session.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    books_data = []

    for x in soup.find_all('a'):
        # это два условия, по которым находятся настоящие ссылки на произведения
        # (иначе иногда попадает всякий мусор)
        if ('tolstoy' in x.attrs['href']) and (x.attrs['href'][0:4] != 'http'):
            # получаем полную ссылку на страницу с книгой
            book_link = 'http://tolstoy-lit.ru' + x.attrs['href']
            book_title = x.text.strip()
            books_data.append({
                'book_title': book_title,
                'book_url': book_link
            })
    return books_data

In [7]:
def get_text_page(url):
    '''
    Получаем текст со страницы.
    На вход подаётся ссылка на станицу.
    На выходе получаем текст.
    '''
    fin_text = []
    req = session.get(url, headers={'User-Agent': ua.random})
    page = req.content
    soup = BeautifulSoup(page, 'html.parser')
    text_p = soup.find_all('p', {'class': 'tab'})
    for t in text_p:
        # немного предобратавыем текст
        t = t.text
        # удаляем всякие дополнительные символы и лишние получившиеся пробелы
        # убираем деление на абзацы
        t = t.replace('\n', ' ')
        t = t.replace('\r', ' ')
        t = ' '.join(t.split())
        if t:
            fin_text.append(t)
    return ' '.join(fin_text)

In [8]:
def get_whole_text(book_url):
    '''
    Получаем текст целого произведения.
    На входе ссылка на книгу, на выходе -- полный текст.
    '''
    req = session.get(book_url, headers={'User-Agent': ua.random})
    page = req.content
    soup = BeautifulSoup(page, 'html.parser')
    book_info = []

    try:
        links = soup.find_all('a')
        # здесь будут уникальные ссылки на части книги. много где они повторяются
        unique_links = {}
        for l in links:
            link = 'http://tolstoy-lit.ru' + l.attrs['href']
            if link not in unique_links:
                # главы и части книги обозначены либо словом ГЛАВА, либо цифрой
                if ('Глава' in l.text):
                    unique_links[link] = l.text.strip()
        # проходимся по каждой части книги и добавляем оттуда текст в общий текст
        for url, chapter in tqdm(unique_links.items(), position=0, leave=True):
            chapter_text = get_text_page(url)
            book_info.append({
                'chapter_name': chapter,
                'chapter_text': chapter_text
            })
            time.sleep(random.uniform(1.5, 2.5))
        if len(unique_links) == 0:
            book_info.append({
                'chapter_name': 'Полный текст книги',
                'chapter_text': get_text_page(book_url)
            })
    # в случае если нет частей или глав: сразу текст со всей страницы
    except:
        raise TimeoutError('В процессе получения текста книги произошла ошибка.')
    return book_info

In [9]:
def get_everything(url, num_texts=10):
    '''
    Объединяет все функции.
    На входе -- исходная, базовая ссылка.
    Ничего не возвращает, но в процессе тексты сохраняются в общий текстовый файл.
    '''
    result = []
    books_data = get_the_books(url)
    if num_texts <= len(books_data):
        for i in tqdm(range(0, num_texts)):
            # выводим ссылку, с которой работаем
            print(books_data[i]['book_url'])
            # получаем целый текст книги
            book_info = get_whole_text(books_data[i]['book_url'])
            for chapter in book_info:
                result.append({
                    'book_url': books_data[i]['book_url'],
                    'book_title': books_data[i]['book_title'],
                    'chapter_name': chapter['chapter_name'],
                    'chapter_text': chapter['chapter_text']
                })
            time.sleep(random.uniform(1.0, 2.0))
    # если вдруг параметр количества текстов задан слишком большим,
    # функция ничего не возвращает, ломается
    else:
        return None
    with open('books_data.json', 'w', encoding='utf-8') as json_file:
        json.dump(result, json_file, ensure_ascii=False, indent=2)

# Получение текстов

In [10]:
# запускаем парсинг текстов
get_everything('http://tolstoy-lit.ru/tolstoy/proza/index.htm', num_texts=1)

  0%|          | 0/1 [00:00<?, ?it/s]

http://tolstoy-lit.ru/tolstoy/proza/anna-karenina/karenina-1-1.htm


100%|██████████| 1/1 [09:21<00:00, 561.32s/it]


Ура, полный текст Анны Карениной (я сократила выдачу только до одного файла, там и так достаточно много всего) и метаинформация скачаны в файл ```books_data.json```!

Теперь эти тексты глав необходимо предобработать, сделать из них полноценный корпус. Это происходит в файле ```preprocessing.ipynb```.